# 05 — Massive Datasets & Schema Engineering: Open Food Facts
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Big Data, and Data Engineering Interviews.*

---

## 📌 Executive Summary & Interview Expectations
In technical interviews for Data Engineering and Production Data Science roles, working with **wide datasets** (100+ columns) and **large-scale files** is a standard test. Interviewers evaluate whether you understand:
1. **The `DtypeWarning` & Chunked Ingestion**: Why Pandas emits mixed-type warnings and what `low_memory=False` actually does under the hood.
2. **Memory Bottlenecks with Massive Files**: Strategies to ingest files larger than RAM (`usecols`, `dtype` specification, chunking via `chunksize`, and PyArrow engines).
3. **The Dangers of `df.values[i][j]`**: Why dropping into raw NumPy 2D arrays causes silent type coercion to `object` and hardcodes fragile column positions.
4. **Safe Positional vs Label Lookups**: Comparing `.iat[i, j]`, `.iloc[i, j]`, and `.loc[label, col]`.
5. **Streaming Chunk Processing**: How to compute global statistics across gigabyte-scale datasets in $O(1)$ memory.

## 1. Environment Setup & Ingesting Wide Files

### ⚠️ Top Interview Question: What does `low_memory=False` actually do?
- **Default Behavior (`low_memory=True`)**:
  - Pandas reads the file in internal buffer chunks (typically 262,144 bytes).
  - It infers the data type for each column *chunk by chunk*.
  - If a column contains only integers in Chunk 1, but contains strings in Chunk 5, Pandas is forced to convert the column to `object` and issues a `DtypeWarning: Columns have mixed types`.
- **Setting `low_memory=False`**:
  - Pandas reads the **entire file into memory** before determining dtypes.
  - This eliminates the warning, but drastically increases RAM consumption!
- **Production Solution**: Specify explicit types upfront via `dtype={...}` or load only necessary columns via `usecols=[...]`.

In [1]:
import os
import numpy as np
import pandas as pd

# Load dataset (tab-delimited, wide schema with 160+ columns)
tsv_path = "en.openfoodfacts.org.products.tsv"
food = pd.read_csv(tsv_path, sep="\t", low_memory=False)
print(f"Data successfully loaded. Dimensions: {food.shape}")

Data successfully loaded. Dimensions: (25, 163)


## 2. High-Dimensional Inspection: Columns & Shape

In [2]:
# First 5 entries
food.head()

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,product_name,generic_name,quantity,...,col_153,col_154,col_155,col_156,col_157,col_158,col_159,col_160,col_161,col_162
0,0,sample,sample,sample,sample,sample,sample,Farine de blé noir,sample,sample,...,sample,sample,sample,sample,sample,sample,sample,sample,sample,sample
1,1,sample,sample,sample,sample,sample,sample,Banana Chips Sweetened (Whole),sample,sample,...,sample,sample,sample,sample,sample,sample,sample,sample,sample,sample
2,2,sample,sample,sample,sample,sample,sample,Peanuts,sample,sample,...,sample,sample,sample,sample,sample,sample,sample,sample,sample,sample
3,3,sample,sample,sample,sample,sample,sample,Organic Salted Nut Mix,sample,sample,...,sample,sample,sample,sample,sample,sample,sample,sample,sample,sample
4,4,sample,sample,sample,sample,sample,sample,Organic Polenta,sample,sample,...,sample,sample,sample,sample,sample,sample,sample,sample,sample,sample


In [3]:
# Number of observations (rows) and columns
print(f"Number of observations (rows): {food.shape[0]:,}")
print(f"Number of features (columns):  {food.shape[1]:,}")

Number of observations (rows): 25
Number of features (columns):  163


In [4]:
# Display column names (truncated for readability)
print("Total Columns:", len(food.columns))
print("First 10 columns:", list(food.columns[:10]))
print("Last 10 columns: ", list(food.columns[-10:]))

Total Columns: 163
First 10 columns: ['code', 'url', 'creator', 'created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'product_name', 'generic_name', 'quantity']
Last 10 columns:  ['col_153', 'col_154', 'col_155', 'col_156', 'col_157', 'col_158', 'col_159', 'col_160', 'col_161', 'col_162']


## 3. Positional Column Lookup & Data Types
In wide datasets, locating features by index position requires zero-based offset indexing.

In [5]:
# Identify the 105th column (index 104, 0-indexed)
col_105_name = food.columns[104]
print(f"105th column name: '{col_105_name}'")

105th column name: '-glucose_100g'


In [6]:
# Check data type of the 105th column
col_105_dtype = food.dtypes[col_105_name]
print(f"Dtype of '{col_105_name}': {col_105_dtype}")

Dtype of '-glucose_100g': float64


In [7]:
# Inspect index
print("Dataset Index Type:", type(food.index))
print("Index Details:     ", food.index)

Dataset Index Type: <class 'pandas.RangeIndex'>
Index Details:      RangeIndex(start=0, stop=25, step=1)


## 4. Retrieving Observations: The Danger of `df.values[i][j]`

In the original exercise, the 19th observation's product name was accessed using:
```python
food.values[18][7]
```

### ⚠️ Top Interview Trap: Why `df.values[i][j]` is an Anti-Pattern
1. **Memory & Type Degradation**:
   - `food.values` extracts the underlying 2D NumPy array.
   - Because NumPy arrays require **homogeneous** data types, if your DataFrame contains a mix of strings, integers, and floats, NumPy forces the **entire array to `object` dtype**!
   - This duplicates the entire DataFrame in memory as boxed Python objects.
2. **Fragile Hardcoded Indices**:
   - Index `[7]` assumes `product_name` is always the 8th column. If someone adds, removes, or reorders columns upstream, `food.values[18][7]` silently returns the wrong column!
3. **Idiomatic Alternatives**:
   - `food.loc[18, 'product_name']`: **Recommended**. Self-documenting, immune to column reordering.
   - `food.iat[18, 7]`: Ultra-fast scalar access (if positional coordinates are strictly required) without copying the full DataFrame to NumPy!

In [8]:
# ❌ Fragile / Anti-pattern:
anti_pattern_val = food.values[18][7]

# ✅ Idiomatic, Robust, Self-Documenting:
robust_val = food.loc[18, "product_name"]

# ✅ Fast scalar positional lookup:
fast_val = food.iat[18, 7]

print(f"Using food.values[18][7]:         '{anti_pattern_val}'")
print(f"Using food.loc[18, 'product_name']: '{robust_val}'")
print(f"Using food.iat[18, 7]:             '{fast_val}'")

Using food.values[18][7]:         'Lotus Organic Brown Jasmine Rice'
Using food.loc[18, 'product_name']: 'Lotus Organic Brown Jasmine Rice'
Using food.iat[18, 7]:             'Lotus Organic Brown Jasmine Rice'


## 5. Big Data & Wide Schema Cheat Sheet

| Challenge | Anti-Pattern | Production Best Practice | Benefit |
| :--- | :--- | :--- | :--- |
| **Mixed Dtypes** | `low_memory=False` | `dtype={'col': str}` or `engine='pyarrow'` | Eliminates warning without RAM bloat |
| **RAM Exhaustion** | Loading all columns | `pd.read_csv(..., usecols=['a', 'b'])` | Reduces RAM usage by up to 95% |
| **Huge Files (> RAM)** | Ingesting in one shot | `pd.read_csv(..., chunksize=100_000)` | Streams file in $O(1)$ memory |
| **Scalar Retrieval** | `df.values[i][j]` | `df.loc[row, 'col']` or `df.iat[i, j]` | Prevents NumPy `object` coercion |
| **Fast Ingestion** | Default C engine | `pd.read_csv(..., engine='pyarrow')` | Multi-threaded SIMD parsing (5-10x faster) |

---
## 🎯 6. Technical Interview Corner: Tricky Questions & Drills

### Q1: The 50GB CSV File on an 8GB Laptop
**Question**: An interviewer asks: *"You have a 50GB CSV file on disk, but your machine only has 8GB of RAM. How do you calculate the total sum and mean of a column without crashing with an Out-Of-Memory (OOM) error?"*

**Answer**:
Use **chunked streaming ingestion** via `pd.read_csv(file, chunksize=N)`:
1. `chunksize` returns an iterator of smaller DataFrames.
2. In each iteration, process only the current chunk and accumulate two running scalars: `total_sum` and `total_count`.
3. Garbage collection automatically frees each chunk as you iterate.
4. Total memory consumed is strictly bounded by the size of a single chunk ($O(1)$ memory).

In [9]:
# Demonstration: Out-of-Core streaming calculation using chunksize
def streaming_mean(file_path, col_name, chunk_size=5):
    total_sum = 0.0
    total_count = 0
    
    # Streams file in small chunks
    for chunk in pd.read_csv(file_path, sep="\t", usecols=[col_name], chunksize=chunk_size):
        clean_chunk = chunk[col_name].dropna()
        total_sum += clean_chunk.sum()
        total_count += len(clean_chunk)
        
    return total_sum / total_count if total_count > 0 else 0.0

col_to_test = "-glucose_100g"
computed_mean = streaming_mean("en.openfoodfacts.org.products.tsv", col_to_test)
print(f"Streaming Mean of '{col_to_test}': {computed_mean:.2f}")

Streaming Mean of '-glucose_100g': 50.54


### Q2: The `usecols` Memory Optimization
**Question**: How does `usecols` optimize performance during CSV ingestion? Does Pandas parse the entire line or skip unused columns?

**Answer**:
1. **Memory**: By passing `usecols=['col1', 'col2']`, Pandas only creates Series objects and allocates memory buffers for the requested columns. For a 163-column dataset where you only need 3 columns, this saves **over 95% of memory**!
2. **Speed**: In the standard C engine, lines are scanned, but memory allocation and type conversion are bypassed for discarded columns, significantly speeding up ingestion.

In [10]:
# Benchmark: Loading only 3 columns vs full dataset
import time

t0 = time.perf_counter()
full_df = pd.read_csv("en.openfoodfacts.org.products.tsv", sep="\t", low_memory=False)
t1 = time.perf_counter()

t2 = time.perf_counter()
subset_df = pd.read_csv(
    "en.openfoodfacts.org.products.tsv", 
    sep="\t", 
    usecols=["code", "product_name", "-glucose_100g"]
)
t3 = time.perf_counter()

full_mem = full_df.memory_usage(deep=True).sum()
sub_mem = subset_df.memory_usage(deep=True).sum()

print(f"Full Dataset Memory (163 cols):   {full_mem:,} bytes")
print(f"Subset Dataset Memory (3 cols):    {sub_mem:,} bytes")
print(f"Memory Savings:                    {((full_mem - sub_mem)/full_mem)*100:.1f}%")

Full Dataset Memory (163 cols):   222,082 bytes
Subset Dataset Memory (3 cols):    2,082 bytes
Memory Savings:                    99.1%


### Q3: PyArrow Engine vs Classic C Engine
**Question**: What advantages does `pd.read_csv(..., engine='pyarrow')` provide over the default C parser in modern Pandas?

**Answer**:
1. **True Multi-Threading**: The classic C parser parses CSV lines on a single thread. The PyArrow engine parses across **all available CPU cores** simultaneously.
2. **Strict / Direct Type Inference**: PyArrow uses zero-copy memory layouts and avoids the chunk-based dtype conflicts that cause `DtypeWarning`.
3. **Speed**: Ingestion of large CSV files is typically **3x to 10x faster**.

In [11]:
# Testing PyArrow engine
try:
    arrow_df = pd.read_csv(
        "en.openfoodfacts.org.products.tsv", 
        sep="\t", 
        engine="pyarrow",
        columns=["code", "product_name"]
    )
    print("PyArrow ingestion successful! Shape:", arrow_df.shape)
except Exception as e:
    print("PyArrow note:", e)

PyArrow note: read_csv() got an unexpected keyword argument 'columns'
